# Jupyter Workflow for Kaggle
This notebook replaces the `%%bash` CLI sequence. It sets up the path, overrides settings if needed, freezes the split natively (giving you access to unlabeled images and selected labels), and executes the stages.

In [ ]:
import sys
import json
import pprint
from pathlib import Path

# Make sure Python can find the "code" module folder
REPO_ROOT = Path("/kaggle/working/TomatoMAP")
if not REPO_ROOT.exists():
    REPO_ROOT = Path().absolute().parent # Fallback for local run depending on folder structure

sys.path.insert(0, str(REPO_ROOT / "code"))

# Import our direct submodules from the code path
from src.experiments.paper1_baseline import freeze_split_once, run_stage
from src.experiments.config import ExperimentConfig


## 1. Split Freezing & Label Filtering
You can optionally pass `selected_labels=[...]` here to only train on a subset of labels. The dataset converter will also save an `unlabeled_images.json` file in the generated `coco_dir` for semi-supervised tasks.

In [ ]:
# Path to the config file you want to use
config_path = REPO_ROOT / "code/configs/paper1/exp01_supervised_yolo_baseline.json"

# Example of using selected_labels to limit to specific classes (e.g. semi-supervised learning targets)
# selected_labels = ["unripe green tomato", "fully ripe"] # Filter by specific class names, None uses all.
selected_labels = None

print(f"Freezing split for config: {config_path}")
result = freeze_split_once(
    config_path=config_path,
    repo_root=REPO_ROOT,
    force=True, # Set to True to regenerate the split on Kaggle
    selected_labels=selected_labels
)

pprint.pprint(result)


## 2. Inspecting Unlabeled Images
The newly collected `unlabeled_images` are exported to the JSON directly in the `coco_dir` output.

In [ ]:
# load the manifest manually if needed to verify
with open(config_path, "r") as f:
    config_dict = json.load(f)
from src.experiments.config import ExperimentConfig
cfg = ExperimentConfig.load(config_path).model_dump()
coco_dir = REPO_ROOT / cfg["paths"]["coco_dir"]
unlabeled_json = coco_dir / "unlabeled_images.json"

if unlabeled_json.exists():
    with open(unlabeled_json, "r") as f:
        unlabeled_imgs = json.load(f)
    print(f"Found {len(unlabeled_imgs)} unlabeled images for semi-supervised tasks.")
    print("Sample:", unlabeled_imgs[:5])
else:
    print("No unlabeled list found.")


## 3. Run Training Stage
Instead of `%%bash`, this natively runs the exact same function.

In [ ]:
print("Starting training stage...")
train_res = run_stage(
    config_path=config_path,
    repo_root=REPO_ROOT,
    stage="train"
)
print("Training Finished:", train_res)


## 4. Run Evaluation Stage

In [ ]:
print("Starting evaluation stage...")
eval_res = run_stage(
    config_path=config_path,
    repo_root=REPO_ROOT,
    stage="eval"
)
print("Evaluation Finished:", eval_res)
